# Does a shallow tree improve sonic-log reconstruction in unseen wells?
Liam Mora — INDE 577

## Investigation contract
We compare a custom shallow regression tree with custom linear regression and a
training-mean baseline. The question is whether nonlinear partitions of gamma ray,
bulk density and neutron porosity improve prediction of observed compressional
sonic slowness (DTC) in wells excluded from training. A petrophysicist could use a
validated reconstruction to prioritize missing-log review. This experiment hides
observed targets for evaluation; it does not establish performance where DTC is
genuinely missing, nor does it identify hydrocarbons.

The unit is one depth observation, but the independence boundary is the well.
The primary comparison metric is MAE in microseconds per foot; RMSE emphasizes
large errors. All displayed evidence uses development wells. The 20 reserved
final-test wells remain excluded, including from model selection.

## Data provenance and coverage
The [FORCE 2020 source repository](https://github.com/bolgebrygg/Force-2020-Machine-Learning-competition)
and [archived record](https://zenodo.org/records/4351156) provide the data under
NLOD 2.0. The local archive was verified November 19, 2026; its SHA256 is recorded
in `configs/reserved_wells.json`. See `docs/FORCE_DATA_CHECK.md` for acquisition
and validation. Place the audited archive at `data/local/force-train.zip` to rerun.

Predictors are GR (API), RHOB (g/cm³) and NPHI (porosity fraction). DTC (µs/ft) is
the target. Well identifiers and depth are used for grouping and diagnostics only.
Lithology is not a predictor. Duplicate well/depth pairs, numeric sentinels and
invalid schema are rejected by the adapter; finite extremes remain in the study.

Split assignment precedes complete-case filtering. The training partition contains
706,823 rows, of which 413,532 qualify for regression; validation contains 240,136,
of which 135,154 qualify. Up to 250 eligible rows per training well are sampled
with seed 577, giving 14,500 rows across 58 wells. All eligible rows in the 20
validation wells are scored. Missing predictors or targets exclude an observation;
we do not impute DTC. Complete-case selection limits applicability.

## Mathematical model and selection design
A regression leaf predicts its training-target mean. For node S,
SSE(S) = sum((y_i − mean(y_S))²). At each node, the tree chooses the feature and
threshold minimizing SSE(left) + SSE(right), then recurses. Mean leaves minimize
squared error; predictions are piecewise constant. Our implementation follows
Lecture 14's regression-tree notebook and sorted-prefix SSE calculations.
Depth and minimum leaf size control complexity; this experiment uses no post-pruning.

The linear model minimizes full mean squared error using batch gradient descent
and training-only standardization, as documented in `docs/LINEAR_REGRESSION.md`.
Trees require no scaling. Both custom models use exactly the same training rows.
The baseline predicts the training mean. Trusted least-squares and squared-error
tree implementations check computation but are not additional custom methods.

The predefined six-candidate grid crosses depths 2, 3, 4 with minimum leaf counts
50 and 100. Selection minimizes pooled validation MAE, breaking ties by lower
depth and then larger leaf size. This uses validation for tuning: the selected
score is not an unbiased final performance estimate. No further grid expansion
is made in this notebook.

## Implementation and verification
Reusable code lives in `SquaredErrorTreeRegressor`, `LectureLinearRegressor`, and
`dsm_project.experiments.force_tree.run_experiment`. The runner loads through the
project data adapter. Analytic splits, threshold equality, stopping constraints,
reference agreement and training-only fitting are checked in
`tests/test_student_tree.py`, `tests/test_force_tree.py` and the linear tests.
The following cell runs the bounded comparison and checks well separation.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/force.toml").is_file())
os.chdir(ROOT)
from dsm_project.data.force import FEATURES
from dsm_project.experiments.force_tree import run_experiment

result = run_experiment()
report = result["report"]
training, validation, models = result["training"], result["validation"], result["models"]
reserved = set(json.loads((ROOT / "configs/reserved_wells.json").read_text())["reserved_test_wells"])
assert not set(training.WELL) & set(validation.WELL)
assert not (set(training.WELL) | set(validation.WELL)) & reserved
print("Archive SHA256:", report["source_sha256"])
print("Training sample SHA256:", report["sample_ids_sha256"])
display(pd.DataFrame({"partition": ["training sample", "validation"],
                      "rows": [len(training), len(validation)],
                      "wells": [training.WELL.nunique(), validation.WELL.nunique()]}))
display(training[[*FEATURES, "DTC"]].describe().T)

## Exploration with a purpose
Target distributions help assess whether a model will face a different prediction
range in validation. The following density plot uses all scored observations in
each partition. It compares distribution shapes, not sample sizes or independent
well frequencies. It is diagnostic only; no rows are removed using this plot.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6), constrained_layout=True)
edges = np.histogram_bin_edges(np.r_[training.DTC, validation.DTC], bins=50)
ax.hist(training.DTC, bins=edges, density=True, histtype="step", linewidth=1.8, label="Training sample")
ax.hist(validation.DTC, bins=edges, density=True, histtype="step", linewidth=1.8, label="Validation")
ax.set(xlabel="Observed DTC (µs/ft)", ylabel="Density", title="Target distributions across whole-well partitions")
ax.legend()
plt.show()

A sampled training distribution and pooled validation distribution weight wells
differently. Similar central ranges do not guarantee similar geology or accurate
tail predictions. These observations motivate both residual and per-well checks.

## Complexity and pooled results
Retaining every candidate makes the selection decision inspectable. The curves
compare training and validation MAE; lower training error alone is not a reason
to choose a model. The horizontal line is the existing linear validation MAE.

In [ ]:
candidates = pd.DataFrame(report["candidates"])
display(candidates.round(4))
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for leaf, group in candidates.groupby("min_samples_leaf"):
    ax.plot(group.max_depth, group.validation_mae, marker="o", label=f"Validation, leaf ≥ {leaf}")
    ax.plot(group.max_depth, group.training_mae, linestyle=":", label=f"Training, leaf ≥ {leaf}")
ax.axhline(report["metrics"]["lecture_linear"]["mae"], color="gray", linestyle="--", label="Linear validation")
ax.set(xlabel="Maximum tree depth", ylabel="MAE (µs/ft)", xticks=[2, 3, 4], title="Six retained complexity candidates")
ax.legend()
plt.show()
display(pd.DataFrame(report["metrics"]).T.rename(columns={"mae": "MAE (µs/ft)", "rmse": "RMSE (µs/ft)"}).round(4))
print("Selected:", report["selected_candidate"]["model"])
print("Maximum reference prediction gap:", report["reference_max_prediction_gap"])

The selected tree has depth 4, minimum leaf size 50, and 16 leaves. Validation
MAE is 9.33 µs/ft versus 10.52 for linear regression and 21.57 for the mean
baseline, about an 11.3% reduction relative to linear regression. RMSE also falls,
from 14.94 to 13.54 µs/ft. The reference tree closely matches the custom tree.
This verifies the computation, not future field performance.

Depth-2 trees underperform linear regression; a tree is not automatically better.
Minimum leaf size 100 happens to have lower training MAE at depth 4 but worse
validation MAE. Greedy split paths change, and the fitting objective is SSE rather
than MAE, so monotonic MAE behavior is not guaranteed.

## Failure analysis: residuals and well variation
Residual means prediction minus observed DTC. A negative residual indicates
underprediction. Hexagonal bins show all validation rows without selecting a
favorable subset; colors indicate counts on a logarithmic scale.

In [ ]:
predictions = {name: models[name].predict(validation.loc[:, FEATURES])
               for name in ["lecture_linear", "selected_tree"]}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True, constrained_layout=True)
for ax, (name, prediction) in zip(axes, predictions.items()):
    hb = ax.hexbin(validation.DTC, prediction - validation.DTC, gridsize=45,
                   bins="log", mincnt=1, cmap="viridis")
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set(xlabel="Observed DTC (µs/ft)", ylabel="Prediction − observation (µs/ft)",
           title="Linear" if name == "lecture_linear" else "Selected shallow tree")
    fig.colorbar(hb, ax=ax, label="Observations (log scale)")
plt.show()
well_mae = result["per_well"].pivot(index="well", columns="model", values="mae")
well_mae["tree_minus_linear"] = well_mae.selected_tree - well_mae.lecture_linear
display(well_mae[["mean_baseline", "lecture_linear", "selected_tree", "tree_minus_linear"]].sort_values("tree_minus_linear").round(3))
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
ordered = well_mae.sort_values("tree_minus_linear")
ax.barh(ordered.index, ordered.tree_minus_linear,
        color=np.where(ordered.tree_minus_linear < 0, "#406b9e", "#c46b22"))
ax.axvline(0, color="gray")
ax.set(xlabel="Tree MAE − linear MAE (µs/ft); negative favors tree", ylabel="Validation well",
       title="Does the pooled gain hold in each well?")
plt.show()
print("Wells improved:", int((well_mae.tree_minus_linear < 0).sum()), "of", len(well_mae))
display(well_mae[["lecture_linear", "selected_tree"]].agg(["mean", "median", "min", "max"]).round(3))
# Post-hoc sensitivity: remove the largest-gain well only for this diagnostic.
# Primary selection and full-validation scores above remain unchanged.
largest_gain_well = well_mae.tree_minus_linear.idxmin()
remaining = result["per_well"].query("well != @largest_gain_well")
sensitivity = []
for name in ["lecture_linear", "selected_tree"]:
    group = remaining.query("model == @name")
    sensitivity.append({"model": name, "remaining_rows": int(group.rows.sum()),
                        "pooled_mae": np.average(group.mae, weights=group.rows)})
print("Post-hoc omission of largest-gain well:", largest_gain_well)
display(pd.DataFrame(sensitivity).round(4))


The residual plots test for systematic errors that a single MAE conceals.
Large negative errors at high observed DTC signal underprediction; piecewise
constant leaf means cannot follow every target extreme. The per-well table retains
both improvements and regressions. The mean of per-well MAEs weights each well
equally, whereas the pooled MAE used for selection weights observations equally.
These answer different questions; neither should replace the other silently.

One fixed development split cannot quantify uncertainty across new geological
settings. Neighboring depths are dependent, so row-wise confidence intervals would
overstate precision. Repeated whole-well splits and rock-type slices remain planned.

### Extrapolation and a controlled feature slice
The next plot varies GR while fixing RHOB and NPHI at their training medians.
The dashed bounds show the training GR range. This is a synthetic diagnostic,
not a physically validated sequence of rocks and not a causal intervention.
It reveals model behavior where input combinations may be poorly represented.

The tree improves 13 of 20 wells and worsens seven. In 25/8-5 S, MAE falls from
24.41 to 6.38 µs/ft across 7,455 observations. The post-hoc sensitivity table
omits this largest-gain well to show how much the pooled advantage depends on it.
This omission was chosen after seeing results: it is neither a new evaluation
split nor grounds for excluding a difficult well from the primary comparison.
Both models still severely underpredict high DTC values, as the common-axis
residual plots show. Further geology-specific review is warranted.


In [ ]:
low, high = training.GR.min(), training.GR.max()
span = high - low
grid = np.linspace(low - .15*span, high + .15*span, 500)
probe = pd.DataFrame({"GR": grid, "RHOB": training.RHOB.median(), "NPHI": training.NPHI.median()})
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for name, label in [("lecture_linear", "Linear"), ("selected_tree", "Shallow tree")]:
    ax.plot(grid, models[name].predict(probe.loc[:, FEATURES]), label=label)
ax.axvline(low, linestyle="--", color="gray", label="Training GR limits")
ax.axvline(high, linestyle="--", color="gray")
ax.set(xlabel="Gamma ray (API)", ylabel="Predicted DTC (µs/ft)", title="Synthetic GR slice at median RHOB and NPHI")
ax.legend()
plt.show()
tree_prediction = predictions["selected_tree"]
assert tree_prediction.min() >= training.DTC.min() - 1e-10
assert tree_prediction.max() <= training.DTC.max() + 1e-10
display(pd.DataFrame({"quantity": ["Training DTC", "Validation DTC", "Tree predictions"],
                      "minimum": [training.DTC.min(), validation.DTC.min(), tree_prediction.min()],
                      "maximum": [training.DTC.max(), validation.DTC.max(), tree_prediction.max()]}))

On this median-feature slice the selected tree is flat throughout the GR range;
GR does not change the reached leaf along this particular path. More generally,
the tree returns constant leaf means beyond its outer split thresholds; the
linear model can continue a trend. Neither behavior guarantees reliable predictions
outside represented feature combinations. Tree predictions stay inside the training
target range, so a lower overall MAE does not eliminate extreme-value failures.

## Conclusion and production decision
The selected shallow tree improves pooled development MAE and RMSE over linear
regression on the fixed whole-well split. Retain both models for further assessment:
the tree adds bounded nonlinear structure, while the linear model supplies a simple
comparison. The selection score is optimistic evidence for the chosen grid winner,
not a claim of validated final performance or readiness for autonomous interpretation.

Complete-case bias, correlated depths, geological shift, unequal well weighting,
greedy-tree instability and extrapolation limits constrain the result. Review
per-well failures before promotion. A future application must expose limitations
and support expert review, not claim oil detection or economic benefit.

Next steps are rock-type diagnostics, repeated whole-well sensitivity analysis and
the planned representation study. Freeze modeling choices before scoring the 20
reserved wells. Product integration and final release evaluation remain pending.

## Reproduce
Run all cells with the DSM Project kernel and the audited local archive available.
The experiment saves candidate and per-well diagnostics in `artifacts/force_tree/`.
Run `uv run --locked pytest tests/test_tree_notebook.py -q` to check the notebook.
The full execution test skips without local data; structural checks still run in CI.
Committed outputs are cleared per the notebook standard. The executed review copy
is `artifacts/force_tree/03-tree-comparison.executed.ipynb`.
See `docs/REGRESSION_TREE.md` for algorithm details and the full comparison table.